# Interest Group Prominence in Congressional Speech

## Interactive Analysis Showcase

This notebook provides an interactive walkthrough of the analysis pipeline, demonstrating:

1. **Data Overview**: Understanding the multi-level dataset structure
2. **Exploratory Analysis**: Patterns in interest group mentions
3. **Statistical Models**: What predicts prominence?
4. **Key Findings**: Actionable insights for researchers and practitioners

---

**Author**: Kaleb Mazurek  
**Repository**: [ThesisPipelineRework](https://github.com/kmazurek95/ThesisPipelineRework)  
**Context**: Master's Thesis in Political Science (Revamped Pipeline)

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Statistical modeling
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

# Project paths (relative to notebooks/ directory)
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'output'

print("Libraries loaded successfully")
print(f"Data directory: data/output/")

---

## 1. Data Overview

The analysis uses a **multi-level data structure** with four levels of aggregation:

| Level | Unit of Analysis | Description |
|-------|------------------|-------------|
| Level 1 | Mention | Individual interest group mentions in congressional text |
| Level 2 | Organization | Aggregated statistics per interest group |
| Level 3 | Politician | Aggregated statistics per Congress member |
| Level 4 | Policy Area | Aggregated statistics per policy domain |

In [ ]:
# Load all datasets
level1 = pd.read_csv(DATA_DIR / 'level1.csv', low_memory=False)
level2 = pd.read_csv(DATA_DIR / 'level2_org.csv')
level3 = pd.read_csv(DATA_DIR / 'level3_politician.csv')
level4 = pd.read_csv(DATA_DIR / 'level4_policy.csv')

# Parse dates
level1['date'] = pd.to_datetime(level1['date'])

# Summary
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Level 1 (Mentions):      {len(level1):>10,} rows × {len(level1.columns):>3} columns")
print(f"Level 2 (Organizations): {len(level2):>10,} rows × {len(level2.columns):>3} columns")
print(f"Level 3 (Politicians):   {len(level3):>10,} rows × {len(level3.columns):>3} columns")
print(f"Level 4 (Policy Areas):  {len(level4):>10,} rows × {len(level4.columns):>3} columns")
print("=" * 60)

DATASET SUMMARY
Level 1 (Mentions):          25,106 rows ×  63 columns
Level 2 (Organizations):      1,679 rows ×  13 columns
Level 3 (Politicians):          490 rows ×   9 columns
Level 4 (Policy Areas):          18 rows ×   7 columns


In [ ]:
# Level 1: Mention-level data sample
print("Level 1 Sample (Mention-Level):")
display(level1[['org_id', 'interest_group', 'prominence_prediction', 'party', 'chamber', 'date']].head(10))

In [ ]:
# Key statistics
stats = {
    'Total Mentions': len(level1),
    'Unique Organizations': level1['org_id'].nunique(),
    'Unique Politicians': level1['bioGuideId'].nunique(),
    'High Prominence (%)': level1['prominence_prediction'].mean() * 100,
    'With Speaker Attribution (%)': level1['bioGuideId'].notna().mean() * 100,
    'With Policy Area (%)': level1['issue_area'].notna().mean() * 100,
    'Date Range': f"{level1['date'].min().date()} to {level1['date'].max().date()}",
}

print("\nKEY STATISTICS")
print("-" * 40)
for key, value in stats.items():
    if isinstance(value, float):
        print(f"{key}: {value:.1f}")
    else:
        print(f"{key}: {value}")

---

## 2. Exploratory Data Analysis

### 2.1 Mentions Over Time

In [ ]:
# Weekly aggregation
level1['week'] = level1['date'].dt.to_period('W')
weekly = level1.groupby('week').agg({
    'org_id': 'count',
    'prominence_prediction': 'mean'
}).reset_index()
weekly.columns = ['week', 'mentions', 'pct_high_prominence']
weekly['week_start'] = weekly['week'].apply(lambda x: x.start_time)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Mention counts
axes[0].fill_between(weekly['week_start'], weekly['mentions'], alpha=0.4, color='steelblue')
axes[0].plot(weekly['week_start'], weekly['mentions'], color='steelblue', linewidth=1.5)
axes[0].set_ylabel('Weekly Mentions', fontsize=12)
axes[0].set_title('Interest Group Mentions in Congressional Record (114th Congress)', fontsize=14, fontweight='bold')
axes[0].axhline(weekly['mentions'].mean(), color='red', linestyle='--', alpha=0.7, label=f'Mean: {weekly["mentions"].mean():.0f}')
axes[0].legend()

# Prominence rate
axes[1].plot(weekly['week_start'], weekly['pct_high_prominence'] * 100, color='coral', linewidth=1.5)
axes[1].axhline(level1['prominence_prediction'].mean() * 100, color='gray', linestyle='--', alpha=0.7, label=f'Overall: {level1["prominence_prediction"].mean()*100:.1f}%')
axes[1].set_ylabel('% High Prominence', fontsize=12)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.show()

### 2.2 Organization Categories

In [ ]:
# Category analysis
cat_stats = level1.groupby('CATEGORY').agg({
    'org_id': 'count',
    'prominence_prediction': 'mean'
}).reset_index()
cat_stats.columns = ['category', 'mentions', 'prominence_rate']
cat_stats = cat_stats.sort_values('mentions', ascending=False).head(15)
cat_stats['category_short'] = cat_stats['category'].str.extract(r'\) (.+)$')[0].fillna(cat_stats['category'])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Mention counts
ax1 = axes[0]
bars1 = ax1.barh(cat_stats['category_short'], cat_stats['mentions'], color='steelblue', alpha=0.8)
ax1.set_xlabel('Number of Mentions', fontsize=12)
ax1.set_title('Top 15 Organization Categories by Mention Count', fontsize=13, fontweight='bold')
ax1.invert_yaxis()

# Add value labels
for bar, val in zip(bars1, cat_stats['mentions']):
    ax1.text(val + 50, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

# Prominence rate
ax2 = axes[1]
colors = plt.cm.RdYlGn(cat_stats['prominence_rate'])
bars2 = ax2.barh(cat_stats['category_short'], cat_stats['prominence_rate'] * 100, color=colors, alpha=0.8)
ax2.axvline(level1['prominence_prediction'].mean() * 100, color='black', linestyle='--', alpha=0.7, label='Overall Mean')
ax2.set_xlabel('% High Prominence', fontsize=12)
ax2.set_title('Prominence Rate by Category', fontsize=13, fontweight='bold')
ax2.invert_yaxis()
ax2.legend()

plt.tight_layout()
plt.show()

### 2.3 Lobbying vs. Prominence

In [ ]:
# Filter organizations with lobbying data
org_data = level2[level2['LOBBYING11'] > 0].copy()
org_data = org_data[org_data['total_mentions'] >= 5]  # At least 5 mentions
org_data['log_lobbying'] = np.log10(org_data['LOBBYING11'] + 1)

print(f"Organizations with lobbying data and 5+ mentions: {len(org_data)}")

# Scatter plot
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(
    org_data['log_lobbying'],
    org_data['avg_prominence'] * 100,
    s=org_data['total_mentions'] * 3,
    c=org_data['total_mentions'],
    cmap='viridis',
    alpha=0.6,
    edgecolors='white',
    linewidths=0.5
)

# Trend line
z = np.polyfit(org_data['log_lobbying'], org_data['avg_prominence'] * 100, 1)
p = np.poly1d(z)
x_line = np.linspace(org_data['log_lobbying'].min(), org_data['log_lobbying'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend (slope = {z[0]:.2f})')

# Colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Mention Count', fontsize=11)

# Correlation
corr = org_data['log_lobbying'].corr(org_data['avg_prominence'])
ax.annotate(f'Correlation: r = {corr:.3f}', xy=(0.05, 0.95), xycoords='axes fraction',
            fontsize=12, fontweight='bold', verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('Log10(Lobbying Expenditure 2011)', fontsize=12)
ax.set_ylabel('% High Prominence Mentions', fontsize=12)
ax.set_title('Lobbying Investment vs. Congressional Prominence', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

### 2.4 Party and Chamber Patterns

In [ ]:
# Filter to mentions with party info
party_data = level1[level1['party'].isin(['D', 'R'])].copy()

# Party comparison
party_stats = party_data.groupby('party').agg({
    'org_id': 'count',
    'prominence_prediction': 'mean'
}).reset_index()
party_stats.columns = ['party', 'mentions', 'prominence_rate']
party_stats['party_name'] = party_stats['party'].map({'D': 'Democrats', 'R': 'Republicans'})

# Chamber comparison
chamber_stats = party_data.groupby('chamber').agg({
    'org_id': 'count',
    'prominence_prediction': 'mean'
}).reset_index()
chamber_stats.columns = ['chamber', 'mentions', 'prominence_rate']
chamber_stats['chamber_name'] = chamber_stats['chamber'].map({'H': 'House', 'S': 'Senate'})

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Party
colors_party = ['steelblue', 'coral']
bars1 = axes[0].bar(party_stats['party_name'], party_stats['prominence_rate'] * 100, color=colors_party, alpha=0.8, edgecolor='black')
axes[0].axhline(level1['prominence_prediction'].mean() * 100, color='gray', linestyle='--', alpha=0.7)
axes[0].set_ylabel('% High Prominence', fontsize=12)
axes[0].set_title('Prominence by Party', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 50)

# Add labels
for bar, val in zip(bars1, party_stats['prominence_rate'] * 100):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

# Chamber
colors_chamber = ['#2ca02c', '#9467bd']
bars2 = axes[1].bar(chamber_stats['chamber_name'], chamber_stats['prominence_rate'] * 100, color=colors_chamber, alpha=0.8, edgecolor='black')
axes[1].axhline(level1['prominence_prediction'].mean() * 100, color='gray', linestyle='--', alpha=0.7)
axes[1].set_ylabel('% High Prominence', fontsize=12)
axes[1].set_title('Prominence by Chamber', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 50)

for bar, val in zip(bars2, chamber_stats['prominence_rate'] * 100):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Print stats
print("\nDetailed Statistics:")
print("\nBy Party:")
print(party_stats.to_string(index=False))
print("\nBy Chamber:")
print(chamber_stats.to_string(index=False))

### 2.5 Policy Area Heatmap

In [ ]:
# Filter to mentions with policy area
policy_data = level1[level1['issue_area_name'].notna()].copy()

# Cross-tabulation: Category × Policy Area
policy_data['category_short'] = policy_data['CATEGORY'].str.extract(r'\) (.+)$')[0].fillna(policy_data['CATEGORY'])

cross_tab = pd.crosstab(
    policy_data['category_short'],
    policy_data['issue_area_name'],
    normalize='columns'
) * 100

# Top categories
top_cats = policy_data['category_short'].value_counts().head(12).index
cross_tab = cross_tab.loc[cross_tab.index.isin(top_cats)]

# Plot
fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(cross_tab, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': '% of Policy Area Mentions'},
            linewidths=0.5)
ax.set_xlabel('Policy Area', fontsize=12)
ax.set_ylabel('Organization Category', fontsize=12)
ax.set_title('Which Interest Group Types Dominate Each Policy Area?', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---

## 3. Statistical Models

### 3.1 Model 1: What Predicts High Prominence? (Mention-Level)

In [ ]:
# Prepare data for Model 1
model1_data = level1[level1['party'].isin(['D', 'R'])].copy()
model1_data['prominent'] = model1_data['prominence_prediction'].astype(int)
model1_data['is_democrat'] = (model1_data['party'] == 'D').astype(int)
model1_data['is_senate'] = (model1_data['chamber'] == 'S').astype(int)
model1_data['log_lobbying'] = np.log1p(model1_data['LOBBYING11'].fillna(0))

# Category dummies
model1_data['category_code'] = model1_data['CATEGORY'].str.extract(r'^\((\d+)\)')[0].fillna('0').astype(int)
model1_data['is_labor'] = model1_data['category_code'].isin([301, 302, 303]).astype(int)
model1_data['is_single_issue'] = model1_data['category_code'].isin([1101, 1102, 1103, 1104]).astype(int)
model1_data['is_trade'] = model1_data['category_code'].isin([204, 205]).astype(int)

# Filter complete cases
vars_needed = ['prominent', 'log_lobbying', 'is_democrat', 'is_senate', 'is_labor', 'is_single_issue', 'is_trade']
model1_df = model1_data[vars_needed].dropna()

print(f"Model 1 sample size: {len(model1_df):,} mentions")

# Fit logistic regression
formula = 'prominent ~ log_lobbying + is_democrat + is_senate + is_labor + is_single_issue + is_trade'
model1 = smf.logit(formula, data=model1_df).fit(disp=0)

print("\n" + "=" * 70)
print("MODEL 1: Mention-Level Logistic Regression")
print("DV: High Prominence (0/1)")
print("=" * 70)
print(model1.summary2().tables[1].to_string())

In [ ]:
# Visualize Model 1 coefficients
coef_df = pd.DataFrame({
    'Variable': model1.params.index[1:],  # Exclude intercept
    'Coefficient': model1.params.values[1:],
    'Std_Error': model1.bse.values[1:],
    'P_Value': model1.pvalues.values[1:]
})
coef_df['Significant'] = coef_df['P_Value'] < 0.05
coef_df['CI_Lower'] = coef_df['Coefficient'] - 1.96 * coef_df['Std_Error']
coef_df['CI_Upper'] = coef_df['Coefficient'] + 1.96 * coef_df['Std_Error']

# Sort by coefficient
coef_df = coef_df.sort_values('Coefficient')

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in coef_df['Coefficient']]
alphas = [0.9 if x else 0.4 for x in coef_df['Significant']]

for i, (_, row) in enumerate(coef_df.iterrows()):
    ax.barh(i, row['Coefficient'], color=colors[i], alpha=alphas[i], edgecolor='black')
    ax.plot([row['CI_Lower'], row['CI_Upper']], [i, i], color='black', linewidth=2)

ax.set_yticks(range(len(coef_df)))
ax.set_yticklabels(coef_df['Variable'])
ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Coefficient (Log-Odds)', fontsize=12)
ax.set_title('Model 1: What Predicts High Prominence?', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 3.2 Model 2: Organization-Level Analysis

In [ ]:
# Prepare data for Model 2
model2_data = level2.copy()
model2_data['log_lobbying'] = np.log1p(model2_data['LOBBYING11'].fillna(0))
model2_data['log_mentions'] = np.log1p(model2_data['total_mentions'])

# Category dummies
model2_data['category_code'] = model2_data['CATEGORY'].str.extract(r'^\((\d+)\)')[0].fillna('0').astype(int)
model2_data['is_labor'] = model2_data['category_code'].isin([301, 302, 303]).astype(int)
model2_data['is_single_issue'] = model2_data['category_code'].isin([1101, 1102, 1103, 1104]).astype(int)
model2_data['is_trade'] = model2_data['category_code'].isin([204, 205]).astype(int)

# Filter to orgs with at least 5 mentions
model2_df = model2_data[model2_data['total_mentions'] >= 5].copy()
model2_df['prominence_dv'] = model2_df['avg_prominence']

print(f"Model 2 sample size: {len(model2_df):,} organizations")

# Fit OLS
formula2 = 'prominence_dv ~ log_lobbying + log_mentions + is_labor + is_single_issue + is_trade'
model2 = smf.ols(formula2, data=model2_df).fit()

print("\n" + "=" * 70)
print("MODEL 2: Organization-Level OLS")
print("DV: Average Prominence Rate")
print("=" * 70)
print(model2.summary2().tables[1].to_string())
print(f"\nR-squared: {model2.rsquared:.3f}")
print(f"Adj. R-squared: {model2.rsquared_adj:.3f}")

### 3.3 Model 3: Politician-Level Analysis

In [ ]:
# Prepare data for Model 3
model3_data = level3.copy()
model3_data['is_democrat'] = (model3_data['party'] == 'D').astype(int)
model3_data['is_senate'] = (model3_data['chamber'] == 'S').astype(int)
model3_data['log_mentions'] = np.log1p(model3_data['total_mentions'])

# Filter to politicians with at least 5 mentions
model3_df = model3_data[model3_data['total_mentions'] >= 5].copy()
model3_df['prominence_dv'] = model3_df['avg_prominence']

print(f"Model 3 sample size: {len(model3_df):,} politicians")

# Fit OLS
formula3 = 'prominence_dv ~ is_democrat + is_senate + log_mentions'
model3 = smf.ols(formula3, data=model3_df).fit()

print("\n" + "=" * 70)
print("MODEL 3: Politician-Level OLS")
print("DV: Average Prominence Rate")
print("=" * 70)
print(model3.summary2().tables[1].to_string())
print(f"\nR-squared: {model3.rsquared:.3f}")
print(f"Adj. R-squared: {model3.rsquared_adj:.3f}")

---

## 4. Key Findings Summary

In [ ]:
# Summary table
findings = pd.DataFrame([
    {
        'Finding': 'Lobbying predicts prominence',
        'Evidence': '+7.1% per log unit (p < 0.001)',
        'Implication': 'Well-resourced groups get more substantive attention'
    },
    {
        'Finding': 'Senators > Representatives',
        'Evidence': '+37% higher odds (p < 0.001)',
        'Implication': 'Senate floor time yields more prominent mentions'
    },
    {
        'Finding': 'Democrats give less prominence',
        'Evidence': '-26% vs Republicans (p < 0.001)',
        'Implication': 'Partisan differences in how groups are discussed'
    },
    {
        'Finding': 'Single-issue groups noticed',
        'Evidence': '+34% higher prominence (p < 0.001)',
        'Implication': 'Focused advocacy leads to substantive discussion'
    },
    {
        'Finding': 'Labor unions get prominence',
        'Evidence': '+14% higher odds (p < 0.01)',
        'Implication': 'Labor issues discussed substantively'
    },
])

print("="*80)
print("KEY FINDINGS: What Drives Interest Group Prominence in Congress?")
print("="*80)
print()
for i, row in findings.iterrows():
    print(f"📊 {row['Finding']}")
    print(f"   Evidence: {row['Evidence']}")
    print(f"   Implication: {row['Implication']}")
    print()

---

## 5. Top Organizations by Prominence

In [ ]:
# Top organizations by prominence (with minimum mentions)
top_orgs = level2[level2['total_mentions'] >= 20].nlargest(15, 'avg_prominence')[[
    'org_name', 'total_mentions', 'avg_prominence', 'CATEGORY'
]].copy()
top_orgs['category_short'] = top_orgs['CATEGORY'].str.extract(r'\) (.+)$')[0].fillna(top_orgs['CATEGORY'])
top_orgs['avg_prominence'] = top_orgs['avg_prominence'] * 100

print("Top 15 Organizations by Prominence (minimum 20 mentions):")
print()
display(top_orgs[['org_name', 'total_mentions', 'avg_prominence', 'category_short']].rename(columns={
    'org_name': 'Organization',
    'total_mentions': 'Mentions',
    'avg_prominence': '% High Prominence',
    'category_short': 'Category'
}))

In [ ]:
# Most mentioned organizations
most_mentioned = level2.nlargest(15, 'total_mentions')[[
    'org_name', 'total_mentions', 'avg_prominence', 'LOBBYING11'
]].copy()
most_mentioned['avg_prominence'] = most_mentioned['avg_prominence'] * 100
most_mentioned['LOBBYING11'] = most_mentioned['LOBBYING11'].apply(lambda x: f'${x/1e6:.1f}M' if pd.notna(x) and x > 0 else 'N/A')

print("Top 15 Most Mentioned Organizations:")
print()
display(most_mentioned.rename(columns={
    'org_name': 'Organization',
    'total_mentions': 'Total Mentions',
    'avg_prominence': '% High Prominence',
    'LOBBYING11': 'Lobbying (2011)'
}))

---

## Conclusion

This analysis reveals systematic patterns in how interest groups receive attention in Congress:

1. **Resources matter**: Groups that invest in lobbying get more substantive discussion
2. **Institution effects**: The Senate provides a more favorable venue for prominent mentions
3. **Party differences**: Democrats and Republicans discuss interest groups differently
4. **Focus pays off**: Single-issue advocacy groups achieve higher prominence rates

These findings have implications for understanding interest group influence, political representation, and the dynamics of congressional discourse.

---

**For more information:**
- Repository: [github.com/kmazurek95/ThesisPipelineRework](https://github.com/kmazurek95/ThesisPipelineRework)
- Contact: [Kaleb Mazurek](https://linkedin.com/in/kalebmazurek)